# Synthetic CDV Experiment - Analysis

This notebook analyzes the multi-seed experiment results across 4 alpha levels.

**Key Questions:**
1. Does CDV matching converge to global performance at alpha=0 (homogeneous)?
2. Does CDV outperform global under heterogeneity (alpha > 0)?
3. How does the advantage scale with alpha?
4. Is the difference statistically significant?

In [ ]:
import os
import json as json_lib
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

from cdv_utils.experiment_runner import load_experiment_results
from cdv_utils.results_analysis import (
    prepare_dataframes_for_analysis,
    calculate_ate_mse_decomposition,
    calculate_ate_by_estimator,
    calculate_ite_mse_pehe,
    calculate_ite_mse_pehe_no_estimator,
    perform_ate_mse_statistical_tests,
    perform_cate_mse_statistical_tests,
    create_comprehensive_statistical_table,
    create_comprehensive_cate_statistical_table,
    add_error_columns,
)
from cdv_utils.visualization import (
    setup_plotting_style,
    plot_ate_bias_variance_tradeoff,
    plot_cate_bias_variance_tradeoff,
    plot_statistical_results_summary,
)

BEST_ESTIMATOR_NAME = 'Best Estimators Selection Per Seed'

setup_plotting_style()
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

# Load config
with open("results/synthetic_experiment_config.json", "r") as f:
    config = json_lib.load(f)

ALPHA_VALUES = config["alpha_values"]
print(f"Alpha values: {ALPHA_VALUES}")
print("Setup complete.")

## 1. Load All Results

In [ ]:
# Load results for all alpha values
results_by_alpha = {}
for alpha in ALPHA_VALUES:
    path = f"results/synthetic_alpha_{alpha:.2f}.pkl"
    if os.path.exists(path):
        results_by_alpha[alpha] = load_experiment_results(path)
        print(f"alpha={alpha:.2f}: loaded {len(results_by_alpha[alpha])} seeds")
    else:
        print(f"alpha={alpha:.2f}: NOT FOUND - run experiment first!")

print(f"\nLoaded {len(results_by_alpha)} alpha levels.")

# Quick sanity check: inspect the structure of one seed result
sample_alpha = ALPHA_VALUES[0]
sample_seed = list(results_by_alpha[sample_alpha].keys())[0]
sample_result = results_by_alpha[sample_alpha][sample_seed]
print(f"\nResult keys per seed: {list(sample_result.keys())}")
for key, val in sample_result.items():
    if isinstance(val, pd.DataFrame):
        print(f"  {key}: DataFrame {val.shape}, columns={list(val.columns)}")
    else:
        print(f"  {key}: {type(val).__name__}")

## 2. Prepare DataFrames per Alpha

Using the same `prepare_dataframes_for_analysis()` function as in Notebook 03, 
we extract and concatenate the per-seed result DataFrames into unified DataFrames 
for each alpha level.


In [ ]:
# Prepare analysis dataframes for each alpha level
analysis_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ALPHA = {alpha}")
    print(f"{'='*70}")
    
    results_by_seed = results_by_alpha[alpha]
    seeds = sorted(results_by_seed.keys())
    
    dataframes = prepare_dataframes_for_analysis(results_by_seed, seeds)
    analysis_by_alpha[alpha] = {
        'dataframes': dataframes,
        'seeds': seeds,
    }

print(f"\nPrepared analysis for {len(analysis_by_alpha)} alpha levels.")

## 3. ATE MSE Decomposition per Alpha (Best Estimator)


In [ ]:
# ATE MSE decomposition (best-estimator) per alpha
ate_decomposition_by_alpha = {}
ate_by_seed_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n--- Alpha = {alpha} ---")
    dfs = analysis_by_alpha[alpha]['dataframes']
    seeds = analysis_by_alpha[alpha]['seeds']
    
    decomp_df, ate_by_seed_df = calculate_ate_mse_decomposition(
        dfs['DF_BEST_GLOBAL'], dfs['DF_BEST_VARIANT'], seeds
    )
    ate_decomposition_by_alpha[alpha] = decomp_df
    ate_by_seed_by_alpha[alpha] = ate_by_seed_df
    
    display(decomp_df)

print("\nATE decomposition complete for all alpha levels.")

## 4. CATE (ITE) MSE Decomposition per Alpha (Best Estimator)


In [ ]:
# CATE (ITE-level) MSE/PEHE and bias-variance decomposition per alpha
cate_summary_by_alpha = {}         # ite_summary_df per alpha
cate_decomposition_by_alpha = {}   # ite_decomposition_df per alpha
cate_decomp_best_by_alpha = {}     # best estimator decomposition

for alpha in ALPHA_VALUES:
    print(f"\n--- Alpha = {alpha} ---")
    dfs = analysis_by_alpha[alpha]['dataframes']
    
    # All estimators
    ite_summary_df, ite_decomposition_df = calculate_ite_mse_pehe(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT']
    )
    cate_summary_by_alpha[alpha] = ite_summary_df
    cate_decomposition_by_alpha[alpha] = ite_decomposition_df
    
    # Best estimator (no estimator column)
    ite_summary_best, ite_decomp_best = calculate_ite_mse_pehe_no_estimator(
        dfs['DF_BEST_GLOBAL'], dfs['DF_BEST_VARIANT'], BEST_ESTIMATOR_NAME
    )
    cate_decomp_best_by_alpha[alpha] = ite_decomp_best

# Build cate_summary_df (for scatter plots) per alpha
cate_scatter_data_by_alpha = {}
for alpha in ALPHA_VALUES:
    wanted_cols = ['method', 'estimator', 'bias_squared', 'variance', 'mse_empirical']
    cate_scatter_df = pd.concat([
        cate_decomposition_by_alpha[alpha][wanted_cols].rename(columns={'mse_empirical': 'mse'}),
        cate_decomp_best_by_alpha[alpha][wanted_cols].rename(columns={'mse_empirical': 'mse'})
    ], ignore_index=True)
    cate_scatter_data_by_alpha[alpha] = cate_scatter_df

print("\nCATE analysis complete for all alpha levels.")

## 5. ATE Per-Estimator Analysis & Statistical Tests


In [ ]:
# Per-estimator ATE decomposition + statistical tests per alpha
ate_stat_by_alpha = {}
ate_scatter_data_by_alpha = {}
estimator_ate_by_seed_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ALPHA = {alpha}")
    print(f"{'='*70}")
    dfs = analysis_by_alpha[alpha]['dataframes']
    seeds = analysis_by_alpha[alpha]['seeds']
    
    # Per-estimator ATE decomposition
    estimator_ate_summary, estimator_ate_by_seed = calculate_ate_by_estimator(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT'], seeds
    )
    estimator_ate_by_seed_by_alpha[alpha] = estimator_ate_by_seed
    
    # Build ate_summary_df for scatter plot (best + all estimators)
    decomp = ate_decomposition_by_alpha[alpha].copy()
    decomp['estimator'] = BEST_ESTIMATOR_NAME
    decomp['bias_squared'] = decomp['bias'] ** 2
    estimator_ate_summary['bias_squared'] = estimator_ate_summary['bias'] ** 2
    
    ate_scatter_df = pd.concat([
        decomp[['estimator', 'method', 'bias_squared', 'variance', 'mse']],
        estimator_ate_summary[['estimator', 'method', 'bias_squared', 'variance', 'mse']]
    ], ignore_index=True)
    ate_scatter_data_by_alpha[alpha] = ate_scatter_df
    
    # Statistical significance (ATE)
    ate_stat_df = perform_ate_mse_statistical_tests(
        estimator_ate_by_seed, estimator_ate_summary,
        df_best_global=dfs['DF_BEST_GLOBAL'],
        df_best_variant=dfs['DF_BEST_VARIANT'],
        best_estimator_name=BEST_ESTIMATOR_NAME
    )
    ate_stat_by_alpha[alpha] = ate_stat_df

## 6. ATE Comprehensive Statistical Tables (Cohen's d, p-values)


In [ ]:
# ATE comprehensive statistical tables per alpha
for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"ATE Statistical Results — Alpha = {alpha}")
    print(f"{'='*70}")
    
    ate_stat_df = ate_stat_by_alpha[alpha]
    if ate_stat_df is not None and not ate_stat_df.empty:
        comprehensive_table = create_comprehensive_statistical_table(ate_stat_df)
        display(comprehensive_table)
    else:
        print("No results available.")

## 7. CATE Statistical Tests & Comprehensive Tables per Alpha


In [ ]:
# CATE statistical tests + comprehensive tables per alpha
cate_stat_by_alpha = {}

for alpha in ALPHA_VALUES:
    print(f"\n{'='*70}")
    print(f"CATE Statistical Results — Alpha = {alpha}")
    print(f"{'='*70}")
    dfs = analysis_by_alpha[alpha]['dataframes']
    
    cate_stat_df = perform_cate_mse_statistical_tests(
        dfs['DF_ALL_GLOBAL'], dfs['DF_ALL_VARIANT'],
        df_best_global=dfs['DF_BEST_GLOBAL'],
        df_best_variant=dfs['DF_BEST_VARIANT']
    )
    cate_stat_by_alpha[alpha] = cate_stat_df
    
    if cate_stat_df is not None and not cate_stat_df.empty:
        comprehensive_cate_table = create_comprehensive_cate_statistical_table(cate_stat_df)
        display(comprehensive_cate_table)
    else:
        print("No results available.")

## 8. Bias²-Variance Scatter Plots per Alpha

These scatter plots show how each estimator trades off bias² vs variance 
for global vs CDV methods. Points closer to origin (0,0) are better.
As alpha increases, CDV points should shift toward lower bias² while 
global points accumulate more bias².


In [ ]:
# ATE bias²-variance scatter plots per alpha
os.makedirs("plots", exist_ok=True)

for alpha in ALPHA_VALUES:
    print(f"\n--- ATE Bias²-Variance Scatter — Alpha = {alpha} ---")
    plot_ate_bias_variance_tradeoff(
        ate_scatter_data_by_alpha[alpha],
        best_estimator_name=BEST_ESTIMATOR_NAME,
        title_prefix=f"ATE (α={alpha})",
        save_path=f"plots/ate_bias_var_scatter_alpha_{alpha:.2f}.png"
    )
    plt.show()

In [ ]:
# CATE bias²-variance scatter plots per alpha
for alpha in ALPHA_VALUES:
    print(f"\n--- CATE Bias²-Variance Scatter — Alpha = {alpha} ---")
    plot_cate_bias_variance_tradeoff(
        cate_scatter_data_by_alpha[alpha],
        best_estimator_name=BEST_ESTIMATOR_NAME,
        title_prefix=f"CATE (α={alpha})",
        save_path=f"plots/cate_bias_var_scatter_alpha_{alpha:.2f}.png"
    )
    plt.show()

## 9. Ranking Metrics per Alpha (Kendall's τ, Spearman's ρ)

Beyond estimation accuracy (MSE), we evaluate how well each method **ranks** individuals 
by predicted treatment effect. Even if a model's ITE estimates are biased, correct ranking 
enables correct prescriptive decisions (treat top-k).

- **Kendall's τ**: Proportion of concordant pairs minus discordant pairs  
- **Spearman's ρ**: Rank correlation between predicted and true ITE


In [ ]:
# Compute ranking metrics (Kendall's τ, Spearman's ρ) per alpha
from scipy.stats import kendalltau, spearmanr

ranking_results = []

for alpha in ALPHA_VALUES:
    dfs = analysis_by_alpha[alpha]['dataframes']
    seeds = analysis_by_alpha[alpha]['seeds']
    
    for method_key, method_label in [('DF_BEST_GLOBAL', 'global'), ('DF_BEST_VARIANT', 'variant')]:
        df = dfs[method_key]
        
        for seed in seeds:
            seed_df = df[df['seed'] == seed] if 'seed' in df.columns else df
            
            ite_real = seed_df['ite_real'].values
            ite_pred = seed_df['ite_pred'].values
            
            # Skip if constant predictions (no ranking possible)
            if np.std(ite_pred) == 0 or np.std(ite_real) == 0:
                continue
            
            tau, tau_p = kendalltau(ite_real, ite_pred)
            rho, rho_p = spearmanr(ite_real, ite_pred)
            
            ranking_results.append({
                'alpha': alpha,
                'method': method_label,
                'seed': seed,
                'kendall_tau': tau,
                'kendall_p': tau_p,
                'spearman_rho': rho,
                'spearman_p': rho_p,
            })

ranking_df = pd.DataFrame(ranking_results)

# Aggregate across seeds
ranking_summary = ranking_df.groupby(['alpha', 'method']).agg(
    kendall_tau_mean=('kendall_tau', 'mean'),
    kendall_tau_std=('kendall_tau', 'std'),
    spearman_rho_mean=('spearman_rho', 'mean'),
    spearman_rho_std=('spearman_rho', 'std'),
    n_seeds=('seed', 'nunique'),
).reset_index()

print("Ranking Metrics Summary (averaged across seeds)")
print("=" * 80)
display(ranking_summary)

# Pivot for easy comparison
for metric_name, mean_col, std_col in [
    ("Kendall's τ", 'kendall_tau_mean', 'kendall_tau_std'),
    ("Spearman's ρ", 'spearman_rho_mean', 'spearman_rho_std'),
]:
    print(f"\n{metric_name} — Global vs CDV")
    pivot = ranking_summary.pivot(index='alpha', columns='method', values=mean_col)
    pivot['CDV_advantage'] = pivot['variant'] - pivot['global']
    display(pivot)

In [ ]:
# Ranking metrics visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
alphas = sorted(ALPHA_VALUES)

for ax_idx, (metric_name, mean_col, std_col) in enumerate([
    ("Kendall's τ", 'kendall_tau_mean', 'kendall_tau_std'),
    ("Spearman's ρ", 'spearman_rho_mean', 'spearman_rho_std'),
]):
    for method, color, marker, ls in [('global', 'tab:red', 's', '--'), ('variant', 'tab:blue', 'o', '-')]:
        method_data = ranking_summary[ranking_summary['method'] == method].sort_values('alpha')
        label = 'Global' if method == 'global' else 'CDV'
        axes[ax_idx].errorbar(
            method_data['alpha'], method_data[mean_col],
            yerr=method_data[std_col],
            label=label, color=color, marker=marker, linestyle=ls,
            linewidth=2, markersize=8, capsize=4
        )
    
    axes[ax_idx].set_xlabel("α (Heterogeneity Level)", fontsize=12)
    axes[ax_idx].set_ylabel(metric_name, fontsize=12)
    axes[ax_idx].set_title(f"{metric_name} vs Heterogeneity", fontsize=13)
    axes[ax_idx].legend(fontsize=11)
    axes[ax_idx].set_xticks(alphas)
    axes[ax_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/ranking_metrics_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/ranking_metrics_synthetic.png")

## 10. Scissors Chart (ATE MSE vs Alpha)


In [ ]:
# Build scissors data from ATE decomposition results
alphas = sorted(ALPHA_VALUES)

cdv_ate_mse = []
global_ate_mse = []
cdv_ate_bias2 = []
global_ate_bias2 = []
cdv_ate_var = []
global_ate_var = []

for alpha in alphas:
    decomp = ate_decomposition_by_alpha[alpha]
    global_row = decomp[decomp['method'] == 'global'].iloc[0]
    variant_row = decomp[decomp['method'] == 'variant'].iloc[0]
    
    global_ate_mse.append(global_row['mse'])
    cdv_ate_mse.append(variant_row['mse'])
    global_ate_bias2.append(global_row['bias_squared'])
    cdv_ate_bias2.append(variant_row['bias_squared'])
    global_ate_var.append(global_row['variance'])
    cdv_ate_var.append(variant_row['variance'])

# Scissors chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ATE MSE
axes[0].plot(alphas, cdv_ate_mse, "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)
axes[0].plot(alphas, global_ate_mse, "s--", label="Global", color="tab:red", linewidth=2, markersize=8)
axes[0].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)
axes[0].set_ylabel("ATE MSE", fontsize=12)
axes[0].set_title("ATE Estimation Error", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].set_xticks(alphas)

# Panel 2: ATE Bias² component
axes[1].plot(alphas, cdv_ate_bias2, "o-", label="CDV Bias²", color="tab:blue", linewidth=2, markersize=8)
axes[1].plot(alphas, global_ate_bias2, "s--", label="Global Bias²", color="tab:red", linewidth=2, markersize=8)
axes[1].set_xlabel("Alpha (Heterogeneity Level)", fontsize=12)
axes[1].set_ylabel("ATE Bias²", fontsize=12)
axes[1].set_title("ATE Bias² Component", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_xticks(alphas)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/scissors_chart_ate_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/scissors_chart_ate_synthetic.png")

## 11. Bias-Variance Decomposition Bars


In [ ]:
# Bias-Variance decomposition bar chart (ATE level)
fig, axes = plt.subplots(1, len(ALPHA_VALUES), figsize=(4*len(ALPHA_VALUES), 5), sharey=True)

for i, alpha in enumerate(alphas):
    x = [0, 1]
    bias2 = [cdv_ate_bias2[i], global_ate_bias2[i]]
    variance = [cdv_ate_var[i], global_ate_var[i]]
    
    axes[i].bar(x, bias2, width=0.4, label="Bias²", color="tab:orange", alpha=0.8)
    axes[i].bar(x, variance, width=0.4, bottom=bias2, label="Variance", color="tab:blue", alpha=0.8)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(["CDV", "Global"])
    axes[i].set_title(f"α = {alpha:.2f}", fontsize=12)
    if i == 0:
        axes[i].set_ylabel("ATE MSE Decomposition", fontsize=11)
    axes[i].legend(fontsize=9)

plt.suptitle("Bias-Variance Decomposition across Heterogeneity Levels", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("plots/bias_variance_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/bias_variance_synthetic.png")

## 12. Summary Tables for Paper


In [ ]:
# Generate summary tables for paper (ATE + CATE + Ranking)
summary_rows = []
for alpha in sorted(ALPHA_VALUES):
    decomp = ate_decomposition_by_alpha[alpha]
    global_row = decomp[decomp['method'] == 'global'].iloc[0]
    variant_row = decomp[decomp['method'] == 'variant'].iloc[0]
    
    # CATE data (best estimator)
    cate_decomp = cate_decomp_best_by_alpha[alpha]
    cate_global = cate_decomp[cate_decomp['method'] == 'global'].iloc[0]
    cate_variant = cate_decomp[cate_decomp['method'] == 'variant'].iloc[0]
    
    # ATE improvement
    ate_imp = ((global_row['mse'] - variant_row['mse']) / global_row['mse'] * 100 
               if global_row['mse'] > 0 else 0)
    
    # CATE improvement
    cate_imp = ((cate_global['mse_empirical'] - cate_variant['mse_empirical']) / cate_global['mse_empirical'] * 100 
                if cate_global['mse_empirical'] > 0 else 0)
    
    # Statistical significance (ATE)
    ate_stat = ate_stat_by_alpha[alpha]
    best_row = ate_stat[ate_stat['Estimator'] == BEST_ESTIMATOR_NAME]
    p_val = best_row['P_Value'].values[0] if len(best_row) > 0 else np.nan
    effect_size = best_row['Effect_Size'].values[0] if len(best_row) > 0 else np.nan
    
    # Ranking metrics
    rank_global = ranking_summary[(ranking_summary['alpha'] == alpha) & (ranking_summary['method'] == 'global')]
    rank_variant = ranking_summary[(ranking_summary['alpha'] == alpha) & (ranking_summary['method'] == 'variant')]
    
    row = {
        "α": f"{alpha:.2f}",
        "ATE MSE (CDV)": variant_row['mse'],
        "ATE MSE (Global)": global_row['mse'],
        "ATE Δ%": ate_imp,
        "CATE MSE (CDV)": cate_variant['mse_empirical'],
        "CATE MSE (Global)": cate_global['mse_empirical'],
        "CATE Δ%": cate_imp,
        "p-value": p_val,
        "Cohen's d": effect_size,
        "τ (CDV)": rank_variant['kendall_tau_mean'].values[0] if len(rank_variant) > 0 else np.nan,
        "τ (Global)": rank_global['kendall_tau_mean'].values[0] if len(rank_global) > 0 else np.nan,
        "Seeds": int(variant_row['n_seeds']),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("Comprehensive Summary Table for Paper")
print("=" * 120)
display(summary_df)

print("\n\nLaTeX format:")
print(summary_df.to_latex(index=False, float_format="%.4f"))

## 13. Four-Panel Comparison Figure


In [ ]:
# 4-panel figure: ATE MSE scissors + Bias² scissors + Bias-Var bars for extreme alphas + CATE decomp
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel (a): ATE MSE scissors
axes[0,0].plot(alphas, cdv_ate_mse, "o-", label="CDV", color="tab:blue", linewidth=2, markersize=8)
axes[0,0].plot(alphas, global_ate_mse, "s--", label="Global", color="tab:red", linewidth=2, markersize=8)
axes[0,0].set_xlabel("α (Heterogeneity)")
axes[0,0].set_ylabel("ATE MSE")
axes[0,0].set_title("(a) ATE MSE vs Heterogeneity")
axes[0,0].legend()
axes[0,0].set_xticks(alphas)

# Panel (b): ATE Bias² scissors
axes[0,1].plot(alphas, cdv_ate_bias2, "o-", label="CDV Bias²", color="tab:blue", linewidth=2, markersize=8)
axes[0,1].plot(alphas, global_ate_bias2, "s--", label="Global Bias²", color="tab:red", linewidth=2, markersize=8)
axes[0,1].set_xlabel("α (Heterogeneity)")
axes[0,1].set_ylabel("Bias²")
axes[0,1].set_title("(b) ATE Bias² vs Heterogeneity")
axes[0,1].legend()
axes[0,1].set_xticks(alphas)

# Panel (c): Bias-Variance bars for all alpha levels
x = np.arange(len(alphas))
width = 0.35
axes[1,0].bar(x - width/2, cdv_ate_mse, width, label="CDV", color="tab:blue", alpha=0.8)
axes[1,0].bar(x + width/2, global_ate_mse, width, label="Global", color="tab:red", alpha=0.8)
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels([f"{a:.2f}" for a in alphas])
axes[1,0].set_xlabel("α")
axes[1,0].set_ylabel("ATE MSE")
axes[1,0].set_title("(c) ATE MSE Comparison")
axes[1,0].legend()

# Panel (d): CDV improvement (%) vs alpha
improvements = []
for i in range(len(alphas)):
    if global_ate_mse[i] > 0:
        imp = (global_ate_mse[i] - cdv_ate_mse[i]) / global_ate_mse[i] * 100
    else:
        imp = 0
    improvements.append(imp)

colors_bar = ['tab:green' if imp > 0 else 'tab:red' for imp in improvements]
axes[1,1].bar(range(len(alphas)), improvements, color=colors_bar, alpha=0.8)
axes[1,1].set_xticks(range(len(alphas)))
axes[1,1].set_xticklabels([f"{a:.2f}" for a in alphas])
axes[1,1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1,1].set_xlabel("α")
axes[1,1].set_ylabel("CDV Improvement (%)")
axes[1,1].set_title("(d) CDV MSE Improvement over Global")

plt.tight_layout()
plt.savefig("plots/four_panel_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plots/four_panel_synthetic.png")

## 14. Key Findings

**Design:**
- Synthetic DGP with 6 sub-groups, α ∈ {0, 0.33, 0.66, 1.0} controlling heterogeneity
- 500 Monte Carlo seeds per α level; K = 4 (3 top variants + 1 others)

**Expected Results:**
- α = 0 (homogeneous): CDV ≈ Global (safe — no harm when heterogeneity is absent)  
- α → 1: CDV MSE << Global MSE (beneficial — captures variant-specific structure)
- Scissors chart shows diverging lines; improvement % grows with α

**Metrics covered:**
- **ATE MSE** (bias² + variance decomposition) — Tables §5, §6
- **CATE/ITE MSE** (PEHE) — Tables §4, §7
- **Statistical significance** — Cohen's d, paired t-test p-values
- **Ranking accuracy** — Kendall's τ, Spearman's ρ (§9)
- **Bias-variance scatter** — Per-estimator tradeoff visualisation (§8)

**Interpretation:**
- CDV reduces **bias** by modelling each variant's causal structure separately
- CDV increases **variance** (smaller per-variant samples), but bias reduction dominates at high α
- Ranking metrics confirm CDV preserves or improves individual-level ordering
